In [1]:
!pip install datasets

In [2]:
from datasets import load_dataset, Dataset
import re
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
dataset_id = 'sinhala-nlp/Sinhala-Corpus'
split_name = 'train'
num_records = 10000

In [5]:
streaming_dataset = load_dataset(dataset_id, split=split_name, streaming=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [6]:
first_n_records_iterable = streaming_dataset.take(num_records)

In [7]:
first_n_records_dataset = Dataset.from_list(list(first_n_records_iterable))

In [8]:
first_n_records_dataset

Dataset({
    features: ['text', 'source'],
    num_rows: 10000
})

In [9]:
first_n_records_dataset[0]

{'text': '03 වන සියවසට අයත් ඉතා ම දුර්ලභ ඝණයේ බුද්ධ ප්රතිමාවක් සොයා ගැනීමට පකිස්ථානයේ පුරාවිද්යාඥයින් කණ්ඩායමක් සමත්ව තිබේ.\nබීබීසී පුවත් සේවය සඳහන් කළේ, කාබන් දින දර්ශකය අනුව මෙම සැතපෙන බුදු පිළිමය දකුණු ආසියානු රටකින් ලද එම ඝණයේ පැරණිත ම පිළිමය හැටියට සැලකෙන බව ය.\nසැතපෙන බුදු පිළිමය හමු වී තිබෙන්නේ, ගාන්ධාරයෙනි.\n‘ගාන්ධාර තොටිල්ල’ යනුවෙන් හැඳින්වෙන වයඹ පකිස්ථානය ඓතිහාසික බෞද්ධ රාජධානියක් වන අතර බුදු දහම ඉන්දියාවේ සිට නැගෙනහිර දෙසට ව්යාප්ත වීමේදී ගාන්ධාර දේශයත් ඊට ඇතුළත් විය.\nපුරාවිද්යාඥයින් පවසන්නේ, පිළිමය අවුරුදු 1800ක් තරම් පැරණි බව ය.\nකැණීම් මගින් තමන් ගොඩ ගන්නා ලද බුදු පිළිමය පිළිසකර කිරීමට අසීරු මට්ටමින් දුර්වල තත්ත්වයක පවතින බව පකිස්ථාන පුරාවිද්යාඥයින් පවසන බව බීබීසී පුවත් සේවය සඳහන් කළේ ය.\nඋණුසුම් පුවත්',
 'source': 'hplt_2'}

In [10]:
from tqdm import tqdm

In [11]:
text = ''
for i in tqdm(range(10000)):
  temp_text = first_n_records_dataset[i]['text']
  text = text + '\n\n' + temp_text

100%|██████████| 10000/10000 [28:39<00:00,  5.82it/s]


In [12]:
len(text.split())

8400096

In [13]:
def remove_non_allowed_content(text: str) -> str:
    # The pattern matches any character that is NOT one of the following:
    # 1. \u0D80-\u0DFF: Sinhala characters (Unicode block)
    # 2. a-zA-Z: English/Latin letters (both cases)
    # 3. 0-9: Numbers
    # 4. \s: Whitespace characters (space, tab, newline)
    # 5. \.,!?:;'"(){}\[\]@#$&*%+-=/\\<>=|_~^: Common punctuation and symbols

    # We use re.UNICODE flag to ensure correct handling of Unicode characters like Sinhala.
    # The character class [^...] means "match any character that is NOT inside this class."
    pattern = r'[^\u0D80-\u0DFF0-9\s\.,!?:;\'"(){}\[\]@#$&*%+-=/\\<>=|_~^]'

    # Replace any matched characters (the unwanted ones) with an empty string
    cleaned_text = re.sub(pattern, '', text, flags=re.UNICODE)

    return cleaned_text

In [14]:
cleaned_output = remove_non_allowed_content(text)

In [15]:
len(cleaned_output.split())

7975318

In [4]:
drive_path = "/content/drive/MyDrive/ICTer_Workshop/sinhala_phrases_filtered.txt"

In [16]:
with open(drive_path, 'w') as f:
    f.write(cleaned_output)